In [28]:
target_episode_id = "AUTOLab+0d4edc83+2023-10-21-19h-02m-09s"

In [29]:
import argparse
import re
import shutil
from functools import partial
from pathlib import Path

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from lerobot.constants import HF_LEROBOT_HOME
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from oxe_utils.configs import OXE_DATASET_CONFIGS, ActionEncoding, StateEncoding
from oxe_utils.transforms import OXE_STANDARDIZATION_TRANSFORMS
from lerobot.datasets.utils import append_jsonlines
np.set_printoptions(precision=2)


import json
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import mediapy
from scipy.spatial.transform import Rotation as R
import cv2
import imageio

In [30]:
args = {
    "raw_dir": Path("/mnt/hwfile/tangyuhang/droid/1.0.0"),
    "local_dir": Path("/mnt/hwfile/tangyuhang/droid/droid_lerobot"),
    "repo_id": "luanqibazao",
    "use_videos": True,
    
}

default_args = {
    "robot_type": None,
    "fps": None,
    "image_writer_process": 5,
    "image_writer_threads": 10,
    
}

raw_dir = args["raw_dir"]
local_dir = args["local_dir"]
repo_id = args["repo_id"]
use_videos = args["use_videos"]

robot_type = default_args["robot_type"]
fps = default_args["fps"]
image_writer_process = default_args["image_writer_process"]
image_writer_threads = default_args["image_writer_threads"]


last_part = raw_dir.name
if re.match(r"^\d+\.\d+\.\d+$", last_part):
    version = last_part
    dataset_name = raw_dir.parent.name
    data_dir = raw_dir.parent.parent
else:
    version = ""
    dataset_name = last_part
    data_dir = raw_dir.parent

if local_dir is None:
    local_dir = Path(HF_LEROBOT_HOME)
local_dir /= f"{dataset_name}_{version}_lerobot"
if local_dir.exists():
    shutil.rmtree(local_dir)

def transform_raw_dataset(episode, dataset_name):
    traj = next(iter(episode["steps"].batch(episode["steps"].cardinality())))

    if dataset_name in OXE_STANDARDIZATION_TRANSFORMS:
        traj = OXE_STANDARDIZATION_TRANSFORMS[dataset_name](traj)

    if dataset_name in OXE_DATASET_CONFIGS:
        state_obs_keys = OXE_DATASET_CONFIGS[dataset_name]["state_obs_keys"]
    else:
        state_obs_keys = [None for _ in range(8)]

    proprio = tf.concat(
        [
            (
                tf.zeros((tf.shape(traj["action"])[0], 1), dtype=tf.float32)  # padding
                if key is None
                else tf.cast(traj["observation"][key], tf.float32)
            )
            for key in state_obs_keys
        ],
        axis=1,
    )

    traj.update(
        {
            "proprio": proprio,
            "task": traj.pop("language_instruction"),
            "action": tf.cast(traj["action"], tf.float32),
        }
    )

    episode["steps"] = traj
    return episode


def generate_features_from_raw(builder: tfds.core.DatasetBuilder, use_videos: bool = True):
    dataset_name = Path(builder.data_dir).parent.name

    state_names = [f"motor_{i}" for i in range(8)]
    if dataset_name in OXE_DATASET_CONFIGS:
        state_encoding = OXE_DATASET_CONFIGS[dataset_name]["state_encoding"]
        if state_encoding == StateEncoding.POS_EULER:
            state_names = ["x", "y", "z", "roll", "pitch", "yaw", "pad", "gripper"]
            if "libero" in dataset_name:
                state_names = ["x", "y", "z", "roll", "pitch", "yaw", "gripper", "gripper"]  # 2D gripper state
        elif state_encoding == StateEncoding.POS_QUAT:
            state_names = ["x", "y", "z", "rx", "ry", "rz", "rw", "gripper"]
        elif state_encoding == StateEncoding.JOINT:
            state_names = [f"motor_{i}" for i in range(7)] + ["gripper"]
            state_obs_keys = OXE_DATASET_CONFIGS[dataset_name]["state_obs_keys"]
            pad_count = state_obs_keys[:-1].count(None)
            state_names[-pad_count - 1 : -1] = ["pad"] * pad_count
            state_names[-1] = "pad" if state_obs_keys[-1] is None else state_names[-1]

    action_names = [f"motor_{i}" for i in range(8)]
    if dataset_name in OXE_DATASET_CONFIGS:
        action_encoding = OXE_DATASET_CONFIGS[dataset_name]["action_encoding"]
        if action_encoding == ActionEncoding.EEF_POS:
            action_names = ["x", "y", "z", "roll", "pitch", "yaw", "gripper"]
        elif action_encoding == ActionEncoding.JOINT_POS:
            action_names = [f"motor_{i}" for i in range(7)] + ["gripper"]

    DEFAULT_FEATURES = {
        "observation.state": {
            "dtype": "float32",
            "shape": (len(state_names),),
            "names": {"motors": state_names},
        },
        "action": {
            "dtype": "float32",
            "shape": (len(action_names),),
            "names": {"motors": action_names},
        },
    }

    obs = builder.info.features["steps"]["observation"]
    features = {
        f"observation.images.{key}": {
            "dtype": "video" if use_videos else "image",
            "shape": value.shape,
            "names": ["height", "width", "rgb"],
        }
        for key, value in obs.items()
        if "depth" not in key and any(x in key for x in ["image", "rgb"])
    }
    return {**features, **DEFAULT_FEATURES}




In [31]:
builder = tfds.builder(dataset_name, data_dir=data_dir, version=version)
features = generate_features_from_raw(builder, use_videos)
filter_fn = lambda e: e["success"] if dataset_name == "kuka" else True
raw_dataset = (
    builder.as_dataset(split="train")
    .filter(filter_fn)
    .map(partial(transform_raw_dataset, dataset_name=dataset_name))
)

In [32]:
path_to_droid_repo = "/mnt/hwfile/tangyuhang/.cache/huggingface/hub/models--KarlP--droid/snapshots/bcb840c3b496533e0adf548a54b51f2f00057837" # TODO: Replace with the path to your DROID repository

# Load the extrinsics
cam2base_extrinsics_path = f"{path_to_droid_repo}/cam2base_extrinsics.json"
with open(cam2base_extrinsics_path, "r") as f:
    cam2base_extrinsics = json.load(f)

# Load the intrinsics
intrinsics_path = f"{path_to_droid_repo}/intrinsics.json"
with open(intrinsics_path, "r") as f:
    intrinsics = json.load(f)

# Load mapping from episode ID to path, then invert
episode_id_to_path_path = f"{path_to_droid_repo}/episode_id_to_path.json"
with open(episode_id_to_path_path, "r") as f:
    episode_id_to_path = json.load(f)
episode_path_to_id = {v: k for k, v in episode_id_to_path.items()}

# Load camera serials
camera_serials_path = f"{path_to_droid_repo}/camera_serials.json"
with open(camera_serials_path, "r") as f:
    camera_serials = json.load(f)

for episode in tqdm(raw_dataset.as_numpy_iterator()):
    file_path = episode["episode_metadata"]["file_path"].decode("utf-8")
    recording_folderpath = episode["episode_metadata"]["recording_folderpath"].decode("utf-8")
    
    episode_path = file_path.split("r2d2-data-full/")[1].split("/trajectory")[0]
    if episode_path not in episode_path_to_id:
        continue
    episode_id = episode_path_to_id[episode_path]        

    if episode_id==target_episode_id:
        break

print(f"episode_id:     {episode_id}")
print(f"target_episode: {target_episode_id}")


3112it [03:22, 15.39it/s]


episode_id:     AUTOLab+0d4edc83+2023-10-21-19h-02m-09s
target_episode: AUTOLab+0d4edc83+2023-10-21-19h-02m-09s


In [33]:
# Iterate through the extrinsics to find key that is a digit
# This is the camera serial number, and the corresponding value is the extrinsics
for k, v in cam2base_extrinsics[episode_id].items():
    if k.isdigit():
        camera_serial = k
        extracted_extrinsics = v
        break

# Also lets us get the intrinsics
extracted_intrinsics = intrinsics[episode_id][camera_serial]

# Using the camera serial, find the corresponding camera name (which is used to determine
# which image stream in the episode to use)
camera_serials_to_name = {v: k for k, v in camera_serials[episode_id].items()}
calib_camera_name = camera_serials_to_name[camera_serial]

if calib_camera_name == "ext1_cam_serial":
    calib_image_name = "exterior_image_1_left"
elif calib_camera_name == "ext2_cam_serial":
    calib_image_name = "exterior_image_2_left"
else:
    raise ValueError(f"Unknown camera name: {calib_camera_name}")

# calib_image_name = "exterior_image_2_left"


print(f"Camera with calibration data: {calib_camera_name} --> {calib_image_name}")

# Convert the extrinsics to a homogeneous transformation matrix
pos = extracted_extrinsics[0:3] # translation
rot_mat = R.from_euler("xyz", extracted_extrinsics[3:6]).as_matrix() # rotation

# Make homogenous transformation matrix
cam_to_base_extrinsics_matrix = np.eye(4)
cam_to_base_extrinsics_matrix[:3, :3] = rot_mat
cam_to_base_extrinsics_matrix[:3, 3] = pos

print(cam_to_base_extrinsics_matrix)

# Convert the intrinsics to a matrix
fx, cx, fy, cy = extracted_intrinsics["cameraMatrix"]
intrinsics_matrix = np.array([
        [fx, 0, cx],
        [0, fy, cy],
        [0, 0, 1]
])
print(intrinsics_matrix)


# Save all observations for the calibrated camera and corresponding gripper positions
images = []
cartesian_poses = []
eq_len = len(episode['steps']['observation']['cartesian_position'])
for index in range(eq_len):
    image = episode['steps']['observation'][calib_image_name][index]
    images.append(image)
    cartesian_pose = episode['steps']['observation']['cartesian_position'][index]
    cartesian_poses.append(cartesian_pose)



# length images x 6
cartesian_poses = np.array(cartesian_poses)
# Remove the rotation and make homogeneous: --> length images x 3 --> length images x 4
cartesian_homogeneous_positions = cartesian_poses[:, :3]
cartesian_homogeneous_positions = np.hstack(
    (cartesian_homogeneous_positions, np.ones((cartesian_homogeneous_positions.shape[0], 1)))
)

# Transpose to support matrix multiplication: --> 4 x length images
gripper_position_base = cartesian_homogeneous_positions.T

# Transform gripper position to camera frame, then remove homogeneous component
base_to_cam_extrinsics_matrix = np.linalg.inv(cam_to_base_extrinsics_matrix)
robot_gripper_position_cam = base_to_cam_extrinsics_matrix @ gripper_position_base
robot_gripper_position_cam = robot_gripper_position_cam[:3] # Now 3 x length images

# Finally, use intrinsics to project the gripper position in camera frame into pixel space
pixel_positions = intrinsics_matrix @ robot_gripper_position_cam[:3]
pixel_positions = pixel_positions[:2] / pixel_positions[2]

# Visualize!
vis_images = []
temp_img_path = f"{path_to_droid_repo}/TEMP.png"

for i, image in enumerate(tqdm(images)):
    if i % 10 != 0:
        continue
    
    fig, axs = plt.subplots(1)
    x, y = pixel_positions[0, i] / 1280 * 320, pixel_positions[1, i] / 720 * 180 # Scale to match image dimensions

    # clip coords
    x = np.clip(x, 0, 320)
    y = np.clip(y, 0, 180)

    axs.imshow(image)
    axs.scatter(x, y, c='red', s=20)
    axs.set_xlim(0, 320)
    axs.set_ylim(180, 0)  # Invert y-axis to match image

    # turn off axes
    axs.axis('off')

    # save the figure, then reopen it as PIL image
    plt.savefig(temp_img_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

    vis_image = Image.open(temp_img_path).convert("RGB")
    vis_images.append(np.array(vis_image))

Camera with calibration data: ext1_cam_serial --> exterior_image_1_left
[[-0.55 -0.27  0.79  0.16]
 [-0.83  0.22 -0.51  0.34]
 [-0.04 -0.94 -0.34  0.45]
 [ 0.    0.    0.    1.  ]]
[[524.26   0.   639.78]
 [  0.   524.26 370.28]
 [  0.     0.     1.  ]]


100%|██████████| 250/250 [00:03<00:00, 76.81it/s]


In [34]:
# Visualize the video
mediapy.show_video(
    vis_images,
    fps=8
)

This browser does not support the video tag.


In [39]:
images[0].shape
pixel_positions[1].max()

np.float64(455.01565274806745)

In [57]:
import json
import numpy as np
import tensorflow_datasets as tfds
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import mediapy
import imageio
from scipy.spatial.transform import Rotation as R


def draw_25d(background_image,pixel_positions, index, draw_len=8):
    
    # length of seq
    episode_len = len(pixel_positions[0])

    # Create a new black image to draw the trajectory on.
    # image=np.zeros((img_height,img_width,3),dtype=np.uint8)
    # demonstration背景
    image = background_image.copy()

    # Store 2D positions of the eef in this episode in a list
    TempProgress = []

    # Store gripper height in a list.
    gripper_height_list = []

    start = index

    if start+draw_len <= episode_len:
        end = start+draw_len
    else:
        end = episode_len

    for i in range(start,end):

        
        u,v = pixel_positions[0, i] / 1280 * 320, pixel_positions[1, i] / 720 * 180 # Scale to match image dimensions

        TempProgress.append((int(u), int(v)))

        # Store the gripper height in a list.
        gripper_height = v
        gripper_height_list.append(gripper_height)

    # Draw gripper height in green color. The higher, the lighter.
    # Draw the Temporal Progress in red color. The earlier the time, the lighter the color of the line segment. Line thickness is 3.
    max_height = max(gripper_height_list)
    min_height = min(gripper_height_list)

    for i in range(1, len(gripper_height_list)):
        # Normalize the gripper height to [0,1]
        if min_height != max_height:
            normalized_gripper_height = float(gripper_height_list[i] - min_height) / (max_height - min_height)
            color = (200, int(255 * normalized_gripper_height), 255 * i / (episode_len - 1))
        else:
            color = (200, 255, 255 * i / (episode_len - 1))
        cv2.line(image, TempProgress[i - 1], TempProgress[i], color=color, thickness=8)

    return image

In [62]:
# Visualize!
vis_images = []
temp_img_path = f"{path_to_droid_repo}/TEMP.png"

for index, image in enumerate(tqdm(images)):
    # if index % 10 != 0:
    #     continue

    image = draw_25d(image,pixel_positions, index, draw_len=32)


    vis_images.append(image)

100%|██████████| 250/250 [00:00<00:00, 5579.67it/s]


In [63]:
# Visualize the video
mediapy.show_video(
    vis_images,
    fps=8
)



This browser does not support the video tag.
